In [1]:
import pandas as pd
import joblib
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from utils import fit_position_pipeline, select_features_by_correlation

In [2]:
# features = joblib.load('../predictors/hist/feats_def')
features = [
    "was_home",
    "difficulty",
    "opponent_difficulty",
    "win_prob",
    "elo_diff",
    "yellow_cards_rolling_5",
    "starts_rolling_3",
    "points_3_ewm",
    "xP_3_ewm",
    "minutes_rolling_3",
    "goals_scored_3_ewm",
    "assists_3_ewm",
    "clean_sheets_3_ewm",
    "goals_conceded_3_ewm",
    "ict_index_3_ewm",
    "expected_goals_3_ewm",
    "expected_assists_3_ewm",
    "expected_goal_involvements_3_ewm",
    "expected_goals_conceded_3_ewm",
    "interceptions_3_ewm",
    "blocks_3_ewm",
    "clearances_3_ewm",
    "tackles_3_ewm",
    "shots_faced_3_ewm",
    "goals_prevented_3_ewm",
    "selected_by_percent",
    "transfers_in",
]

player_data = pd.read_csv("../../inputs/rolled_data_24-25_xp_fixed.csv")
player_data = player_data[player_data["position"] == "Defender"]

league_models = {}

for gw in range(6, 39):
    print(f"\n{'=' * 60}\n Model for round {gw} \n{'=' * 60}")
    data = player_data[(player_data["round"] > 3) & (player_data["round"] < gw)].fillna(
        0
    )
    feats = features  # features[gw]
    # feats.remove('xP')
    selected_feats = select_features_by_correlation(
        data,
        feats,
        target_col="xP",
        redundancy_threshold=0.4,
        min_corr=0.2,
        verbose=True,
    )

    league_model = fit_position_pipeline(
        data, selected_feats, gw, position_name="DEF", random_seed=42
    )

    print(f"Features for DEF kept in gw {gw} -> {league_model['feats']}")

    league_models[gw] = league_model



 Model for round 19 
  dropping 'xP_3_ewm' (|corr| with xP=0.803) -- |corr|=0.838 with already-selected 'points_3_ewm'
  dropping 'ict_index_3_ewm' (|corr| with xP=0.722) -- |corr|=0.795 with already-selected 'points_3_ewm'
  dropping 'minutes_rolling_3' (|corr| with xP=0.691) -- |corr|=0.759 with already-selected 'points_3_ewm'
  dropping 'starts_rolling_3' (|corr| with xP=0.677) -- |corr|=0.744 with already-selected 'points_3_ewm'
  dropping 'clean_sheets_3_ewm' (|corr| with xP=0.641) -- |corr|=0.806 with already-selected 'points_3_ewm'
  dropping 'expected_goal_involvements_3_ewm' (|corr| with xP=0.573) -- |corr|=0.627 with already-selected 'points_3_ewm'
  dropping 'clearances_3_ewm' (|corr| with xP=0.567) -- |corr|=0.633 with already-selected 'points_3_ewm'
  dropping 'transfers_in' (|corr| with xP=0.545) -- |corr|=0.567 with already-selected 'points_3_ewm'
  dropping 'tackles_3_ewm' (|corr| with xP=0.541) -- |corr|=0.594 with already-selected 'points_3_ewm'
  dropping 'expected_

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

... (192 outputs left out)

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  divergences: 0
Features for DEF kept in gw 38 -> ['elo_diff', 'yellow_cards_rolling_5', 'points_3_ewm', 'goals_conceded_3_ewm']


In [3]:
joblib.dump(league_models, "./league_models_def")

['./league_models_def']